<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day5_5_(260529)_Spring_Security_Crud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 금융 트랜젝션 관리

In [ ]:
%%writefile spring-lab/simple-crud/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-jdbc'
    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'
    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting spring-lab/simple-crud/build.gradle


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/testdb
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.thymeleaf.cache=false

Overwriting spring-lab/simple-crud/src/main/resources/application.properties


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/domain/Account.java

package com.example.demo.domain;

import java.time.LocalDateTime;

/**
 * [Domain 계층]
 * accounts 테이블의 한 행을 Java 객체로 표현하는 클래스이다.
 */
public class Account {

    private Long id;
    private String accountNumber;
    private String ownerName;
    private Long balance;
    private LocalDateTime createdAt;

    public Account(
            Long id,
            String accountNumber,
            String ownerName,
            Long balance,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.accountNumber = accountNumber;
        this.ownerName = ownerName;
        this.balance = balance;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getOwnerName() {
        return ownerName;
    }

    public Long getBalance() {
        return balance;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/domain/Account.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/domain/AccountTransaction.java

package com.example.demo.domain;

import java.time.LocalDateTime;

/**
 * [Domain 계층]
 * account_transactions 테이블의 한 행을 Java 객체로 표현하는 클래스이다.
 */
public class AccountTransaction {

    private Long id;
    private String accountNumber;
    private String transactionType;
    private Long amount;
    private String memo;
    private LocalDateTime createdAt;

    public AccountTransaction(
            Long id,
            String accountNumber,
            String transactionType,
            Long amount,
            String memo,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.accountNumber = accountNumber;
        this.transactionType = transactionType;
        this.amount = amount;
        this.memo = memo;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getTransactionType() {
        return transactionType;
    }

    public Long getAmount() {
        return amount;
    }

    public String getMemo() {
        return memo;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/domain/AccountTransaction.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/dto/TransactionForm.java

package com.example.demo.dto;

/**
 * [DTO 계층]
 * 화면에서 입력한 금융 거래 정보를 Controller로 전달하는 객체이다.
 */
public class TransactionForm {

    private String accountNumber;
    private String transactionType;
    private Long amount;
    private String memo;

    public TransactionForm() {
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getTransactionType() {
        return transactionType;
    }

    public Long getAmount() {
        return amount;
    }

    public String getMemo() {
        return memo;
    }

    public void setAccountNumber(String accountNumber) {
        this.accountNumber = accountNumber;
    }

    public void setTransactionType(String transactionType) {
        this.transactionType = transactionType;
    }

    public void setAmount(Long amount) {
        this.amount = amount;
    }

    public void setMemo(String memo) {
        this.memo = memo;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/dto/TransactionForm.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/repository/TransactionRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.dto.TransactionForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class TransactionRepository {

    private final JdbcTemplate jdbcTemplate;

    public TransactionRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Account> accountRowMapper = (rs, rowNum) -> new Account(
            rs.getLong("id"),
            rs.getString("account_number"),
            rs.getString("owner_name"),
            rs.getLong("balance"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    private final RowMapper<AccountTransaction> transactionRowMapper = (rs, rowNum) -> new AccountTransaction(
            rs.getLong("id"),
            rs.getString("account_number"),
            rs.getString("transaction_type"),
            rs.getLong("amount"),
            rs.getString("memo"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public List<Account> findAllAccounts() {
        String sql = """
                SELECT id, account_number, owner_name, balance, created_at
                FROM accounts
                ORDER BY id
                """;

        return jdbcTemplate.query(sql, accountRowMapper);
    }

    public List<AccountTransaction> findAllTransactions() {
        String sql = """
                SELECT id, account_number, transaction_type, amount, memo, created_at
                FROM account_transactions
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, transactionRowMapper);
    }

    public Account findAccountByAccountNumber(String accountNumber) {
        String sql = """
                SELECT id, account_number, owner_name, balance, created_at
                FROM accounts
                WHERE account_number = ?
                """;

        return jdbcTemplate.queryForObject(sql, accountRowMapper, accountNumber);
    }

    public void saveTransaction(TransactionForm form) {
        String sql = """
                INSERT INTO account_transactions
                (account_number, transaction_type, amount, memo)
                VALUES (?, ?, ?, ?)
                """;

        jdbcTemplate.update(
                sql,
                form.getAccountNumber(),
                form.getTransactionType(),
                form.getAmount(),
                form.getMemo()
        );
    }

    public void increaseBalance(String accountNumber, Long amount) {
        String sql = """
                UPDATE accounts
                SET balance = balance + ?
                WHERE account_number = ?
                """;

        jdbcTemplate.update(sql, amount, accountNumber);
    }

    public void decreaseBalance(String accountNumber, Long amount) {
        String sql = """
                UPDATE accounts
                SET balance = balance - ?
                WHERE account_number = ?
                """;

        jdbcTemplate.update(sql, amount, accountNumber);
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/repository/TransactionRepository.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/service/TransactionService.java

package com.example.demo.service;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.dto.TransactionForm;
import com.example.demo.repository.TransactionRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class TransactionService {

    private final TransactionRepository repository;

    public TransactionService(TransactionRepository repository) {
        this.repository = repository;
    }

    public List<Account> findAllAccounts() {
        return repository.findAllAccounts();
    }

    public List<AccountTransaction> findAllTransactions() {
        return repository.findAllTransactions();
    }

    @Transactional
    public void createTransaction(TransactionForm form) {
        validateTransactionForm(form);

        Account account = repository.findAccountByAccountNumber(form.getAccountNumber());

        if ("DEPOSIT".equals(form.getTransactionType())) {
            repository.saveTransaction(form);
            repository.increaseBalance(form.getAccountNumber(), form.getAmount());
            return;
        }

        if ("WITHDRAW".equals(form.getTransactionType())) {
            if (account.getBalance() < form.getAmount()) {
                throw new IllegalArgumentException("잔액이 부족합니다.");
            }

            repository.saveTransaction(form);
            repository.decreaseBalance(form.getAccountNumber(), form.getAmount());
            return;
        }

        throw new IllegalArgumentException("지원하지 않는 거래 유형입니다.");
    }

    private void validateTransactionForm(TransactionForm form) {
        if (form.getAccountNumber() == null || form.getAccountNumber().isBlank()) {
            throw new IllegalArgumentException("계좌번호가 필요합니다.");
        }

        if (form.getTransactionType() == null || form.getTransactionType().isBlank()) {
            throw new IllegalArgumentException("거래 유형이 필요합니다.");
        }

        if (form.getAmount() == null || form.getAmount() <= 0) {
            throw new IllegalArgumentException("거래 금액은 1원 이상이어야 합니다.");
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/service/TransactionService.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/controller/TransactionController.java

package com.example.demo.controller;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.dto.TransactionForm;
import com.example.demo.service.TransactionService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@Controller
public class TransactionController {

    private final TransactionService service;

    public TransactionController(TransactionService service) {
        this.service = service;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/transactions";
    }

    @GetMapping("/transactions")
    public String list(Model model) {
        List<Account> accounts = service.findAllAccounts();
        List<AccountTransaction> transactions = service.findAllTransactions();

        model.addAttribute("accounts", accounts);
        model.addAttribute("transactions", transactions);
        model.addAttribute("transactionForm", new TransactionForm());

        return "transactions";
    }

    @PostMapping("/transactions")
    public String create(@ModelAttribute TransactionForm form, Model model) {
        try {
            service.createTransaction(form);
            return "redirect:/transactions";
        } catch (IllegalArgumentException e) {
            List<Account> accounts = service.findAllAccounts();
            List<AccountTransaction> transactions = service.findAllTransactions();

            model.addAttribute("accounts", accounts);
            model.addAttribute("transactions", transactions);
            model.addAttribute("transactionForm", form);
            model.addAttribute("errorMessage", e.getMessage());

            return "transactions";
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/controller/TransactionController.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/templates/transactions.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>금융 트랜젝션 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container">
    <h1>금융 트랜젝션 관리</h1>

    <section class="card">
        <h2>계좌 목록</h2>

        <table>
            <thead>
            <tr>
                <th>계좌번호</th>
                <th>예금주</th>
                <th>잔액</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="account : ${accounts}">
                <td th:text="${account.accountNumber}">100-111</td>
                <td th:text="${account.ownerName}">김도현</td>
                <td th:text="${#numbers.formatInteger(account.balance, 0, 'COMMA')}">1,000,000</td>
            </tr>
            </tbody>
        </table>
    </section>

    <section class="card">
        <h2>거래 등록</h2>

        <div class="error-box" th:if="${errorMessage != null}" th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/transactions" method="post" th:object="${transactionForm}">
            <div class="form-row">
                <label>계좌번호</label>
                <select th:field="*{accountNumber}" required>
                    <option value="">계좌 선택</option>
                    <option th:each="account : ${accounts}"
                            th:value="${account.accountNumber}"
                            th:text="${account.accountNumber + ' / ' + account.ownerName}">
                        100-111 / 김도현
                    </option>
                </select>
            </div>

            <div class="form-row">
                <label>거래 유형</label>
                <select th:field="*{transactionType}" required>
                    <option value="">거래 유형 선택</option>
                    <option value="DEPOSIT">입금</option>
                    <option value="WITHDRAW">출금</option>
                </select>
            </div>

            <div class="form-row">
                <label>금액</label>
                <input type="number" th:field="*{amount}" placeholder="금액 입력" min="1" required>
            </div>

            <div class="form-row">
                <label>메모</label>
                <input type="text" th:field="*{memo}" placeholder="거래 메모 입력">
            </div>

            <button type="submit">거래 등록</button>
        </form>
    </section>

    <section class="card">
        <h2>거래 내역</h2>

        <table>
            <thead>
            <tr>
                <th>ID</th>
                <th>계좌번호</th>
                <th>거래 유형</th>
                <th>금액</th>
                <th>메모</th>
                <th>등록 시각</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="transaction : ${transactions}">
                <td th:text="${transaction.id}">1</td>
                <td th:text="${transaction.accountNumber}">100-111</td>
                <td th:text="${transaction.transactionType}">DEPOSIT</td>
                <td th:text="${#numbers.formatInteger(transaction.amount, 0, 'COMMA')}">100,000</td>
                <td th:text="${transaction.memo}">입금</td>
                <td th:text="${#temporals.format(transaction.createdAt, 'yyyy-MM-dd HH:mm:ss')}">
                    2026-05-26 10:00:00
                </td>
            </tr>
            </tbody>
        </table>
    </section>
</div>
</body>
</html>

Writing spring-lab/simple-crud/src/main/resources/templates/transactions.html


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f6f8;
    color: #222;
}

.container {
    width: 1000px;
    margin: 40px auto;
}

h1 {
    margin-bottom: 24px;
}

.card {
    background: #ffffff;
    padding: 24px;
    margin-bottom: 24px;
    border: 1px solid #ddd;
    border-radius: 10px;
}

.card h2 {
    margin-top: 0;
}

.form-row {
    margin-bottom: 14px;
}

label {
    display: block;
    margin-bottom: 6px;
    font-weight: bold;
}

input,
select {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 6px;
}

button {
    padding: 9px 16px;
    border: 0;
    border-radius: 6px;
    background: #222;
    color: white;
    cursor: pointer;
}

button:hover {
    background: #000;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th,
td {
    padding: 12px;
    border-bottom: 1px solid #ddd;
    text-align: left;
}

th {
    background: #f0f1f3;
}

.error-box {
    padding: 12px;
    margin-bottom: 16px;
    border: 1px solid #d93025;
    border-radius: 6px;
    background: #fff4f4;
    color: #b00020;
    font-weight: bold;
}

Writing spring-lab/simple-crud/src/main/resources/static/style.css


# Spring Security 기반 로그인/회원가입 및 금융 트랜젝션 관리

In [ ]:
%%writefile spring-lab/simple-crud/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-jdbc'
    implementation 'org.springframework.boot:spring-boot-starter-security'

    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
    testImplementation 'org.springframework.security:spring-security-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting spring-lab/simple-crud/build.gradle


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/testdb
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.thymeleaf.cache=false

Overwriting spring-lab/simple-crud/src/main/resources/application.properties


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/domain/AppUser.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class AppUser {

    private Long id;
    private String email;
    private String password;
    private String name;
    private String role;
    private LocalDateTime createdAt;

    public AppUser(
            Long id,
            String email,
            String password,
            String name,
            String role,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.email = email;
        this.password = password;
        this.name = name;
        this.role = role;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public String getRole() {
        return role;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}


Writing spring-lab/simple-crud/src/main/java/com/example/demo/domain/AppUser.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/domain/Account.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class Account {

    private Long id;
    private Long userId;
    private String accountNumber;
    private String ownerName;
    private Long balance;
    private LocalDateTime createdAt;

    public Account(
            Long id,
            Long userId,
            String accountNumber,
            String ownerName,
            Long balance,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.userId = userId;
        this.accountNumber = accountNumber;
        this.ownerName = ownerName;
        this.balance = balance;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public Long getUserId() {
        return userId;
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getOwnerName() {
        return ownerName;
    }

    public Long getBalance() {
        return balance;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}


Writing spring-lab/simple-crud/src/main/java/com/example/demo/domain/Account.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/domain/AccountTransaction.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class AccountTransaction {

    private Long id;
    private String accountNumber;
    private String transactionType;
    private Long amount;
    private String memo;
    private LocalDateTime createdAt;

    public AccountTransaction(
            Long id,
            String accountNumber,
            String transactionType,
            Long amount,
            String memo,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.accountNumber = accountNumber;
        this.transactionType = transactionType;
        this.amount = amount;
        this.memo = memo;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getTransactionType() {
        return transactionType;
    }

    public Long getAmount() {
        return amount;
    }

    public String getMemo() {
        return memo;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/domain/AccountTransaction.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/dto/SignupForm.java

package com.example.demo.dto;

public class SignupForm {

    private String email;
    private String password;
    private String name;

    public SignupForm() {
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setPassword(String password) {
        this.password = password;
    }

    public void setName(String name) {
        this.name = name;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/dto/SignupForm.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/dto/TransactionForm.java

package com.example.demo.dto;

public class TransactionForm {

    private String accountNumber;
    private String transactionType;
    private Long amount;
    private String memo;

    public TransactionForm() {
    }

    public String getAccountNumber() {
        return accountNumber;
    }

    public String getTransactionType() {
        return transactionType;
    }

    public Long getAmount() {
        return amount;
    }

    public String getMemo() {
        return memo;
    }

    public void setAccountNumber(String accountNumber) {
        this.accountNumber = accountNumber;
    }

    public void setTransactionType(String transactionType) {
        this.transactionType = transactionType;
    }

    public void setAmount(Long amount) {
        this.amount = amount;
    }

    public void setMemo(String memo) {
        this.memo = memo;
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/dto/TransactionForm.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/repository/UserRepository.java

package com.example.demo.repository;

import com.example.demo.domain.AppUser;
import com.example.demo.dto.SignupForm;
import org.springframework.dao.EmptyResultDataAccessException;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.Optional;

@Repository
public class UserRepository {

    private final JdbcTemplate jdbcTemplate;

    public UserRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<AppUser> userRowMapper = (rs, rowNum) -> new AppUser(
            rs.getLong("id"),
            rs.getString("email"),
            rs.getString("password"),
            rs.getString("name"),
            rs.getString("role"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public Optional<AppUser> findByEmail(String email) {
        String sql = """
                SELECT id, email, password, name, role, created_at
                FROM users
                WHERE email = ?
                """;

        try {
            AppUser user = jdbcTemplate.queryForObject(sql, userRowMapper, email);
            return Optional.of(user);
        } catch (EmptyResultDataAccessException e) {
            return Optional.empty();
        }
    }

    public void save(SignupForm form, String encodedPassword) {
        String sql = """
                INSERT INTO users (email, password, name, role)
                VALUES (?, ?, ?, 'USER')
                """;

        jdbcTemplate.update(
                sql,
                form.getEmail(),
                encodedPassword,
                form.getName()
        );
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/repository/UserRepository.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/repository/TransactionRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.dto.TransactionForm;
import org.springframework.dao.EmptyResultDataAccessException;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;
import java.util.Optional;

@Repository
public class TransactionRepository {

    private final JdbcTemplate jdbcTemplate;

    public TransactionRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Account> accountRowMapper = (rs, rowNum) -> new Account(
            rs.getLong("id"),
            rs.getLong("user_id"),
            rs.getString("account_number"),
            rs.getString("owner_name"),
            rs.getLong("balance"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    private final RowMapper<AccountTransaction> transactionRowMapper = (rs, rowNum) -> new AccountTransaction(
            rs.getLong("id"),
            rs.getString("account_number"),
            rs.getString("transaction_type"),
            rs.getLong("amount"),
            rs.getString("memo"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public List<Account> findAccountsByUserId(Long userId) {
        String sql = """
                SELECT id, user_id, account_number, owner_name, balance, created_at
                FROM accounts
                WHERE user_id = ?
                ORDER BY id
                """;

        return jdbcTemplate.query(sql, accountRowMapper, userId);
    }

    public List<AccountTransaction> findTransactionsByUserId(Long userId) {
        String sql = """
                SELECT t.id, t.account_number, t.transaction_type, t.amount, t.memo, t.created_at
                FROM account_transactions t
                JOIN accounts a ON t.account_number = a.account_number
                WHERE a.user_id = ?
                ORDER BY t.id DESC
                """;

        return jdbcTemplate.query(sql, transactionRowMapper, userId);
    }

    public Optional<Account> findAccountByAccountNumberAndUserId(String accountNumber, Long userId) {
        String sql = """
                SELECT id, user_id, account_number, owner_name, balance, created_at
                FROM accounts
                WHERE account_number = ?
                  AND user_id = ?
                """;

        try {
            Account account = jdbcTemplate.queryForObject(
                    sql,
                    accountRowMapper,
                    accountNumber,
                    userId
            );

            return Optional.of(account);
        } catch (EmptyResultDataAccessException e) {
            return Optional.empty();
        }
    }

    public void saveTransaction(TransactionForm form) {
        String sql = """
                INSERT INTO account_transactions
                (account_number, transaction_type, amount, memo)
                VALUES (?, ?, ?, ?)
                """;

        jdbcTemplate.update(
                sql,
                form.getAccountNumber(),
                form.getTransactionType(),
                form.getAmount(),
                form.getMemo()
        );
    }

    public void increaseBalance(String accountNumber, Long amount) {
        String sql = """
                UPDATE accounts
                SET balance = balance + ?
                WHERE account_number = ?
                """;

        jdbcTemplate.update(sql, amount, accountNumber);
    }

    public void decreaseBalance(String accountNumber, Long amount) {
        String sql = """
                UPDATE accounts
                SET balance = balance - ?
                WHERE account_number = ?
                """;

        jdbcTemplate.update(sql, amount, accountNumber);
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/repository/TransactionRepository.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/service/CustomUserDetailsService.java

package com.example.demo.service;

import com.example.demo.domain.AppUser;
import com.example.demo.repository.UserRepository;
import org.springframework.security.core.authority.SimpleGrantedAuthority;
import org.springframework.security.core.userdetails.User;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class CustomUserDetailsService implements UserDetailsService {

    private final UserRepository repository;

    public CustomUserDetailsService(UserRepository repository) {
        this.repository = repository;
    }

    @Override
    public UserDetails loadUserByUsername(String email) throws UsernameNotFoundException {
        AppUser user = repository.findByEmail(email)
                .orElseThrow(() -> new UsernameNotFoundException("사용자를 찾을 수 없습니다."));

        return new User(
                user.getEmail(),
                user.getPassword(),
                List.of(new SimpleGrantedAuthority("ROLE_" + user.getRole()))
        );
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/service/CustomUserDetailsService.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/service/UserService.java

package com.example.demo.service;

import com.example.demo.domain.AppUser;
import com.example.demo.dto.SignupForm;
import com.example.demo.repository.UserRepository;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Service;

@Service
public class UserService {

    private final UserRepository repository;
    private final PasswordEncoder passwordEncoder;

    public UserService(
            UserRepository repository,
            PasswordEncoder passwordEncoder
    ) {
        this.repository = repository;
        this.passwordEncoder = passwordEncoder;
    }

    public void signup(SignupForm form) {
        validateSignupForm(form);

        if (repository.findByEmail(form.getEmail()).isPresent()) {
            throw new IllegalArgumentException("이미 가입된 이메일입니다.");
        }

        String encodedPassword = passwordEncoder.encode(form.getPassword());

        repository.save(form, encodedPassword);
    }

    public AppUser findByEmail(String email) {
        return repository.findByEmail(email)
                .orElseThrow(() -> new IllegalArgumentException("사용자를 찾을 수 없습니다."));
    }

    private void validateSignupForm(SignupForm form) {
        if (form.getEmail() == null || form.getEmail().isBlank()) {
            throw new IllegalArgumentException("이메일이 필요합니다.");
        }

        if (form.getPassword() == null || form.getPassword().isBlank()) {
            throw new IllegalArgumentException("비밀번호가 필요합니다.");
        }

        if (form.getName() == null || form.getName().isBlank()) {
            throw new IllegalArgumentException("이름이 필요합니다.");
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/service/UserService.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/service/TransactionService.java

package com.example.demo.service;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.dto.TransactionForm;
import com.example.demo.repository.TransactionRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class TransactionService {

    private final TransactionRepository repository;

    public TransactionService(TransactionRepository repository) {
        this.repository = repository;
    }

    public List<Account> findAccounts(Long userId) {
        return repository.findAccountsByUserId(userId);
    }

    public List<AccountTransaction> findTransactions(Long userId) {
        return repository.findTransactionsByUserId(userId);
    }

    @Transactional
    public void createTransaction(Long userId, TransactionForm form) {
        validateTransactionForm(form);

        Account account = repository.findAccountByAccountNumberAndUserId(
                form.getAccountNumber(),
                userId
        ).orElseThrow(() -> new IllegalArgumentException("본인 계좌만 거래할 수 있습니다."));

        if ("DEPOSIT".equals(form.getTransactionType())) {
            repository.saveTransaction(form);
            repository.increaseBalance(form.getAccountNumber(), form.getAmount());
            return;
        }

        if ("WITHDRAW".equals(form.getTransactionType())) {
            if (account.getBalance() < form.getAmount()) {
                throw new IllegalArgumentException("잔액이 부족합니다.");
            }

            repository.saveTransaction(form);
            repository.decreaseBalance(form.getAccountNumber(), form.getAmount());
            return;
        }

        throw new IllegalArgumentException("지원하지 않는 거래 유형입니다.");
    }

    private void validateTransactionForm(TransactionForm form) {
        if (form.getAccountNumber() == null || form.getAccountNumber().isBlank()) {
            throw new IllegalArgumentException("계좌번호가 필요합니다.");
        }

        if (form.getTransactionType() == null || form.getTransactionType().isBlank()) {
            throw new IllegalArgumentException("거래 유형이 필요합니다.");
        }

        if (form.getAmount() == null || form.getAmount() <= 0) {
            throw new IllegalArgumentException("거래 금액은 1원 이상이어야 합니다.");
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/service/TransactionService.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/config/SecurityConfig.java

package com.example.demo.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;

@Configuration
public class SecurityConfig {

    @Bean
    public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
        http
                .authorizeHttpRequests(auth -> auth
                        .requestMatchers("/signup", "/login", "/style.css").permitAll()
                        .requestMatchers("/admin").hasRole("ADMIN")
                        .requestMatchers("/transactions").authenticated()
                        .anyRequest().authenticated()
                )
                .formLogin(form -> form
                        .loginPage("/login")
                        .loginProcessingUrl("/login")
                        .defaultSuccessUrl("/transactions", true)
                        .permitAll()
                )
                .logout(logout -> logout
                        .logoutUrl("/logout")
                        .logoutSuccessUrl("/login?logout")
                        .permitAll()
                );

        return http.build();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/config/SecurityConfig.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/controller/AuthController.java

package com.example.demo.controller;

import com.example.demo.dto.SignupForm;
import com.example.demo.service.UserService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

@Controller
public class AuthController {

    private final UserService service;

    public AuthController(UserService service) {
        this.service = service;
    }

    @GetMapping("/login")
    public String login() {
        return "login";
    }

    @GetMapping("/signup")
    public String signupForm(Model model) {
        model.addAttribute("signupForm", new SignupForm());
        return "signup";
    }

    @PostMapping("/signup")
    public String signup(@ModelAttribute SignupForm form, Model model) {
        try {
            service.signup(form);
            return "redirect:/login?signup";
        } catch (IllegalArgumentException e) {
            model.addAttribute("signupForm", form);
            model.addAttribute("errorMessage", e.getMessage());
            return "signup";
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/controller/AuthController.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/controller/TransactionController.java

package com.example.demo.controller;

import com.example.demo.domain.Account;
import com.example.demo.domain.AccountTransaction;
import com.example.demo.domain.AppUser;
import com.example.demo.dto.TransactionForm;
import com.example.demo.service.TransactionService;
import com.example.demo.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@Controller
public class TransactionController {

    private final TransactionService transactionService;
    private final UserService userService;

    public TransactionController(
            TransactionService transactionService,
            UserService userService
    ) {
        this.transactionService = transactionService;
        this.userService = userService;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/transactions";
    }

    @GetMapping("/transactions")
    public String list(Authentication authentication, Model model) {
        AppUser user = userService.findByEmail(authentication.getName());

        List<Account> accounts = transactionService.findAccounts(user.getId());
        List<AccountTransaction> transactions = transactionService.findTransactions(user.getId());

        model.addAttribute("loginUser", user);
        model.addAttribute("accounts", accounts);
        model.addAttribute("transactions", transactions);
        model.addAttribute("transactionForm", new TransactionForm());

        return "transactions";
    }

    @PostMapping("/transactions")
    public String create(
            Authentication authentication,
            @ModelAttribute TransactionForm form,
            Model model
    ) {
        AppUser user = userService.findByEmail(authentication.getName());

        try {
            transactionService.createTransaction(user.getId(), form);
            return "redirect:/transactions";
        } catch (IllegalArgumentException e) {
            List<Account> accounts = transactionService.findAccounts(user.getId());
            List<AccountTransaction> transactions = transactionService.findTransactions(user.getId());

            model.addAttribute("loginUser", user);
            model.addAttribute("accounts", accounts);
            model.addAttribute("transactions", transactions);
            model.addAttribute("transactionForm", form);
            model.addAttribute("errorMessage", e.getMessage());

            return "transactions";
        }
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/controller/TransactionController.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/java/com/example/demo/controller/AdminController.java

package com.example.demo.controller;

import org.springframework.stereotype.Controller;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
public class AdminController {

    @GetMapping("/admin")
    public String admin() {
        return "admin";
    }
}

Writing spring-lab/simple-crud/src/main/java/com/example/demo/controller/AdminController.java


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/templates/login.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>로그인</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container narrow">
    <h1>로그인</h1>

    <section class="card">
        <div class="success-box" th:if="${param.signup}">
            회원가입이 완료되었습니다. 로그인해 주세요.
        </div>

        <div class="success-box" th:if="${param.logout}">
            로그아웃되었습니다.
        </div>

        <div class="error-box" th:if="${param.error}">
            이메일 또는 비밀번호가 올바르지 않습니다.
        </div>

        <form action="/login" method="post">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <div class="form-row">
                <label>이메일</label>
                <input type="email" name="username" placeholder="이메일 입력" required>
            </div>

            <div class="form-row">
                <label>비밀번호</label>
                <input type="password" name="password" placeholder="비밀번호 입력" required>
            </div>

            <button type="submit">로그인</button>
            <a href="/signup" class="link-button">회원가입</a>
        </form>
    </section>
</div>
</body>
</html>

Writing spring-lab/simple-crud/src/main/resources/templates/login.html


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/templates/signup.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>회원가입</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container narrow">
    <h1>회원가입</h1>

    <section class="card">
        <div class="error-box" th:if="${errorMessage != null}" th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/signup" method="post" th:object="${signupForm}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <div class="form-row">
                <label>이메일</label>
                <input type="email" th:field="*{email}" placeholder="이메일 입력" required>
            </div>

            <div class="form-row">
                <label>비밀번호</label>
                <input type="password" th:field="*{password}" placeholder="비밀번호 입력" required>
            </div>

            <div class="form-row">
                <label>이름</label>
                <input type="text" th:field="*{name}" placeholder="이름 입력" required>
            </div>

            <button type="submit">회원가입</button>
            <a href="/login" class="link-button">로그인</a>
        </form>
    </section>
</div>
</body>
</html>

Writing spring-lab/simple-crud/src/main/resources/templates/signup.html


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/templates/transactions.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>금융 트랜젝션 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container">
    <div class="top-bar">
        <div>
            <h1>금융 트랜젝션 관리</h1>
            <p th:text="${loginUser.name + '님 로그인 중'}">사용자 로그인 중</p>
        </div>

        <form action="/logout" method="post">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
            <button type="submit" class="secondary-button">로그아웃</button>
        </form>
    </div>

    <section class="card">
        <h2>내 계좌 목록</h2>

        <table>
            <thead>
            <tr>
                <th>계좌번호</th>
                <th>예금주</th>
                <th>잔액</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="account : ${accounts}">
                <td th:text="${account.accountNumber}">100-111</td>
                <td th:text="${account.ownerName}">김도현</td>
                <td th:text="${#numbers.formatInteger(account.balance, 0, 'COMMA')}">1,000,000</td>
            </tr>
            </tbody>
        </table>
    </section>

    <section class="card">
        <h2>거래 등록</h2>

        <div class="error-box" th:if="${errorMessage != null}" th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/transactions" method="post" th:object="${transactionForm}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <div class="form-row">
                <label>계좌번호</label>
                <select th:field="*{accountNumber}" required>
                    <option value="">계좌 선택</option>
                    <option th:each="account : ${accounts}"
                            th:value="${account.accountNumber}"
                            th:text="${account.accountNumber + ' / ' + account.ownerName}">
                        100-111 / 김도현
                    </option>
                </select>
            </div>

            <div class="form-row">
                <label>거래 유형</label>
                <select th:field="*{transactionType}" required>
                    <option value="">거래 유형 선택</option>
                    <option value="DEPOSIT">입금</option>
                    <option value="WITHDRAW">출금</option>
                </select>
            </div>

            <div class="form-row">
                <label>금액</label>
                <input type="number" th:field="*{amount}" placeholder="금액 입력" min="1" required>
            </div>

            <div class="form-row">
                <label>메모</label>
                <input type="text" th:field="*{memo}" placeholder="거래 메모 입력">
            </div>

            <button type="submit">거래 등록</button>
        </form>
    </section>

    <section class="card">
        <h2>내 거래 내역</h2>

        <table>
            <thead>
            <tr>
                <th>ID</th>
                <th>계좌번호</th>
                <th>거래 유형</th>
                <th>금액</th>
                <th>메모</th>
                <th>등록 시각</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="transaction : ${transactions}">
                <td th:text="${transaction.id}">1</td>
                <td th:text="${transaction.accountNumber}">100-111</td>
                <td th:text="${transaction.transactionType}">DEPOSIT</td>
                <td th:text="${#numbers.formatInteger(transaction.amount, 0, 'COMMA')}">100,000</td>
                <td th:text="${transaction.memo}">입금</td>
                <td th:text="${#temporals.format(transaction.createdAt, 'yyyy-MM-dd HH:mm:ss')}">
                    2026-05-26 10:00:00
                </td>
            </tr>
            </tbody>
        </table>
    </section>
</div>
</body>
</html>

Writing spring-lab/simple-crud/src/main/resources/templates/transactions.html


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/templates/admin.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>관리자 화면</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container narrow">
    <h1>관리자 화면</h1>

    <section class="card">
        <p>ADMIN 권한 사용자만 접근할 수 있는 화면입니다.</p>

        <a href="/transactions" class="link-button">거래 화면으로 이동</a>
    </section>
</div>
</body>
</html>

Writing spring-lab/simple-crud/src/main/resources/templates/admin.html


In [ ]:
%%writefile spring-lab/simple-crud/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f6f8;
    color: #222;
}

.container {
    width: 1000px;
    margin: 40px auto;
}

.container.narrow {
    width: 480px;
}

h1 {
    margin-bottom: 24px;
}

.card {
    background: #ffffff;
    padding: 24px;
    margin-bottom: 24px;
    border: 1px solid #ddd;
    border-radius: 10px;
}

.card h2 {
    margin-top: 0;
}

.form-row {
    margin-bottom: 14px;
}

label {
    display: block;
    margin-bottom: 6px;
    font-weight: bold;
}

input,
select {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 6px;
}

button,
.link-button {
    display: inline-block;
    padding: 9px 16px;
    border: 0;
    border-radius: 6px;
    background: #222;
    color: white;
    cursor: pointer;
    text-decoration: none;
    font-size: 14px;
}

button:hover,
.link-button:hover {
    background: #000;
}

.secondary-button {
    background: #555;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th,
td {
    padding: 12px;
    border-bottom: 1px solid #ddd;
    text-align: left;
}

th {
    background: #f0f1f3;
}

.error-box {
    padding: 12px;
    margin-bottom: 16px;
    border: 1px solid #d93025;
    border-radius: 6px;
    background: #fff4f4;
    color: #b00020;
    font-weight: bold;
}

.success-box {
    padding: 12px;
    margin-bottom: 16px;
    border: 1px solid #1e8e3e;
    border-radius: 6px;
    background: #f0fff4;
    color: #137333;
    font-weight: bold;
}

.top-bar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 24px;
}

.top-bar h1 {
    margin-bottom: 8px;
}

.top-bar p {
    margin: 0;
    color: #555;
}

Writing spring-lab/simple-crud/src/main/resources/static/style.css
